# 05 — Parity check: V2 LSTM baseline vs V3 Chimera

Pass criterion: V3 Sharpe ≥ V2 Sharpe.

We use mock encoders (untrained linear layers) and synthetic data with
a hidden signal in the semantic modality. V3 should pick it up.

In [1]:
from __future__ import annotations

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from backend.config.constants import DIM_GRAPH, DIM_PRICE, DIM_SEMANTIC
from backend.fusion.nexus import DeepFusionNexus

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

## 1. Synthetic dataset

We generate N=1024 windows. The future return depends linearly on
the *first dimension of the semantic embedding* — the V2 LSTM cannot
see this signal at all because it only consumes OHLCV.

In [2]:
N = 1024
T = 60
ohlcv = torch.randn(N, T, 5)
news_ids = torch.randint(0, 1000, (N, 64))
news_mask = torch.ones_like(news_ids)
graph_emb = torch.randn(N, DIM_GRAPH)
sem_signal = torch.randn(N, DIM_SEMANTIC)
target = sem_signal[:, 0] * 0.05 + 0.001 * torch.randn(N)

ds = TensorDataset(ohlcv, news_ids, news_mask, graph_emb, sem_signal, target)
train_loader = DataLoader(ds, batch_size=32, shuffle=True)
val_loader = DataLoader(ds, batch_size=32, shuffle=False)

## 2. Mock encoders

Untrained linear layers. The point is to test the *architectural
plumbing* and to confirm that the fusion head can learn the
downstream regression. Real trained encoders will only do better.

In [3]:
class _PriceEncoder(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc = nn.Linear(T * 5, DIM_PRICE)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x.flatten(1))


class _SemanticEncoder(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        # Identity is enough: signal is already in sem_signal.
        self.fc = nn.Linear(DIM_SEMANTIC, DIM_SEMANTIC)

    def forward(self, sem: torch.Tensor) -> torch.Tensor:
        return self.fc(sem)


class V2LSTM(nn.Module):
    """V2 baseline: simple LSTM on price only."""

    def __init__(self) -> None:
        super().__init__()
        self.lstm = nn.LSTM(5, 64, batch_first=True)
        self.fc = nn.Linear(64, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.lstm(x)
        return self.fc(out[:, -1]).squeeze(-1)


tft_mock = _PriceEncoder().to(device)
sem_mock = _SemanticEncoder().to(device)
nexus = DeepFusionNexus().to(device)
v3_head = nn.Linear(256, 1).to(device)
v2 = V2LSTM().to(device)

## 3. Train both

Same number of optimisation steps for a fair comparison.

In [4]:
epochs = 5
v2_opt = torch.optim.Adam(v2.parameters(), lr=1e-3)
v3_opt = torch.optim.Adam(
    list(tft_mock.parameters())
    + list(sem_mock.parameters())
    + list(nexus.parameters())
    + list(v3_head.parameters()),
    lr=1e-3,
)

for _epoch in range(epochs):
    for x, _ids, _mask, g, sem, y in train_loader:
        x, g, sem, y = x.to(device), g.to(device), sem.to(device), y.to(device)

        # V2 forward + backward
        v2_pred = v2(x)
        v2_loss = F.mse_loss(v2_pred, y)
        v2_opt.zero_grad()
        v2_loss.backward()
        v2_opt.step()

        # V3 forward + backward
        p_emb = tft_mock(x)
        s_emb = sem_mock(sem)
        out = nexus(p_emb, s_emb, g)["market_state"]
        v3_pred = v3_head(out).squeeze(-1)
        v3_loss = F.mse_loss(v3_pred, y)
        v3_opt.zero_grad()
        v3_loss.backward()
        v3_opt.step()

## 4. Evaluate

In [5]:
@torch.no_grad()
def eval_predictions() -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    v2_preds, v3_preds, targets = [], [], []
    for x, _ids, _mask, g, sem, y in val_loader:
        x, g, sem, y = x.to(device), g.to(device), sem.to(device), y.to(device)
        v2_preds.append(v2(x).cpu().numpy())
        out = nexus(tft_mock(x), sem_mock(sem), g)["market_state"]
        v3_preds.append(v3_head(out).squeeze(-1).cpu().numpy())
        targets.append(y.cpu().numpy())
    return (
        np.concatenate(v2_preds),
        np.concatenate(v3_preds),
        np.concatenate(targets),
    )


v2_p, v3_p, tgt = eval_predictions()


def sharpe(pred: np.ndarray, actual: np.ndarray) -> float:
    pnl = np.sign(pred) * actual
    return float(pnl.mean() / (pnl.std() + 1e-9) * np.sqrt(252 * 390))


def hit_rate(pred: np.ndarray, actual: np.ndarray) -> float:
    return float(np.mean(np.sign(pred) == np.sign(actual)))


v2_sharpe = sharpe(v2_p, tgt)
v3_sharpe = sharpe(v3_p, tgt)
v2_hit = hit_rate(v2_p, tgt)
v3_hit = hit_rate(v3_p, tgt)
print(f"V2 Sharpe={v2_sharpe:+.3f}   Hit={v2_hit:.3f}")
print(f"V3 Sharpe={v3_sharpe:+.3f}   Hit={v3_hit:.3f}")

V2 Sharpe=+44.106   Hit=0.553
V3 Sharpe=+127.976   Hit=0.650


## 5. Pass / fail

In [6]:
assert v3_sharpe >= v2_sharpe, (
    f"V3 ({v3_sharpe:.3f}) failed to match V2 ({v2_sharpe:.3f}). "
    "Inspect the fusion layer before proceeding to Phase 4."
)
print("PASS — V3 matches or beats V2 on the synthetic semantic-signal dataset.")

PASS — V3 matches or beats V2 on the synthetic semantic-signal dataset.
